# Quantization harness
Thin runner for Colab / Kaggle (T4). All logic lives in the repo scripts; this notebook only clones, installs and calls them. Set the runtime to a T4 GPU first. Optional secrets: `HF_TOKEN` (gated models), `GITHUB_TOKEN` (push results).

In [ ]:
import os
def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

for k in ("HF_TOKEN", "GITHUB_TOKEN"):
    v = secret(k)
    if v:
        os.environ[k] = v
    print(k, "set" if v else "not set")   # never print the value

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/AbhigyanD/Quantization_Harness.git
%cd Quantization_Harness
!git config user.name "Abhigyan Dey" && git config user.email "abhigyandey2@gmail.com"

## 1. Quantize, measure, evaluate
Runs every stage for fp16, nf4 and gptq. A failing step is logged to `results/<slug>/pipeline_failures.log` and the rest continues.

In [ ]:
!pip install -q -r requirements.txt -r requirements-quant.txt
!pip freeze > results/env_pipeline_freeze.txt
!bash scripts/run_pipeline.sh

In [ ]:
!bash scripts/push_results.sh "Save pipeline results from T4 run"

## 2. vLLM serving benchmark
Uses its own venv because vLLM pins a different torch.

In [ ]:
!bash scripts/run_bench.sh

In [ ]:
!bash scripts/push_results.sh "Save vLLM benchmark results from T4 run"